# Project Overview

## Extract & Load

> The source data is in loose CSV files outside any database; this stage loads the files into DuckDB's `raw` schema. This is the 'EL' in ELT; no cleaning is done at this stage; it is done in the next stage, within the database using SQL.



### Connect to DuckDB; create the schemas

In [1]:
# Import libraries & dependencies
import duckdb
import pandas as pd
from pathlib import Path

# Mount drive and specify path
try:
  # Running in Colab
  from google.colab import drive
  drive.mount('/content/drive/')
  project_dir = Path('/content/drive/MyDrive/Agriculture_ELT')
  db_path = Path('/content/agric_elt.duckdb')
except ImportError:
  # Running locally
  project_dir = Path('..')
  db_path = project_dir / 'data' / 'agric_elt.duckdb'

source_dir = project_dir / 'data'

print('Source folder exists:', source_dir.exists())
print(sorted(p.name for p in source_dir.glob("*.csv")))

Mounted at /content/drive/
Source folder exists: True
['farms.csv', 'harvests.csv', 'inputs.csv', 'plantings.csv', 'weather.csv']


In [2]:
# Connect to db and create schemas
conn = duckdb.connect(str(db_path))

conn.sql("CREATE SCHEMA IF NOT EXISTS raw")
conn.sql("CREATE SCHEMA IF NOT EXISTS staging")
conn.sql("CREATE SCHEMA IF NOT EXISTS marts")

conn.sql(
    "SELECT schema_name from information_schema.schemata WHERE schema_name IN ('raw', 'staging', 'marts')"
).df()

,schema_name
0,marts
1,raw
2,staging


### Load CSV files into `raw` as text.

> Read every column as `VARCHAR` to avoid altering data or load failure; keeping all records as text ensures nothing is lost or altered. Data types will be transformed appropriately in the next stage.
> Two columns are added to each table: source_file (where each row came from) and loaded_at (when it was loaded).




In [3]:
tables = ['farms', 'plantings', 'inputs', 'harvests', 'weather']

for name in tables:
  conn.sql(
      f"""
      CREATE OR REPLACE TABLE raw.{name} AS
      SELECT
        *,
        '{name}.csv' AS source_file,
        current_timestamp AS loaded_at
      FROM read_csv("{source_dir / (name + '.csv')}", header=true, all_varchar=true)
      """
  )

  print(f"loaded raw.{name}")

loaded raw.farms
loaded raw.plantings
loaded raw.inputs
loaded raw.harvests
loaded raw.weather


### Check & Validate Loading Stage

> Check every table's row count matches the no. of rows in corresponding CSV.



In [4]:
checks = []

for name in tables:
  in_db = conn.sql(f"SELECT COUNT(*) FROM raw.{name}").fetchone()[0]
  in_csv = len(pd.read_csv(source_dir / f'{name}.csv', dtype=str))
  checks.append(
      {
          'table': name,
          'rows_in_db': in_db,
          'rows_in_csv': in_csv,
          'match': in_db == in_csv
      }
  )

result = pd.DataFrame(checks)
assert result['match'].all(), "Row count differ between CSV and db!"
result

,table,rows_in_db,rows_in_csv,match
0,farms,812,812,True
1,plantings,5377,5377,True
2,inputs,7009,7009,True
3,harvests,5014,5014,True
4,weather,19078,19078,True


### Preview the raw data

In [ ]:
for name in tables:
  print(f'raw.{name}')
  display(conn.sql(f"SELECT * EXCLUDE (source_file, loaded_at) FROM raw.{name} LIMIT 5").df())
  print('\n')

raw.farms


,farm_id,county,size_acres,soil_type,irrigated,registration_date
0,F-0001,Kakamega,0.6 acres,clay loam,no,29/03/2020
1,F0002,Uasin Gishu,11.1,loam,N,2012-05-06
2,F0003,Meru,1.0,clay,N,2021-04-14
3,F0004,Meru,1.3,Sandy,N,13/11/2008
4,F0005,Narok,6.2,clay loam,No,2021-03-29




raw.plantings


,planting_id,farm_id,season,crop,planting_date,seed_variety,area_planted_acres
0,P00001,f0001,2020-LR,Maize,2020-03-28,H513,0.9
1,P00002,F0001,2023 Long Rains,Beans,"Apr 24, 2023",Local,0.3
2,P00003,F0001,2024 LR,Maize,18/04/2024,LOCAL,0.6
3,P00004,F0002,LR 2019,Maize,2019-03-14,DK8031,8.2
4,P00005,F0002,SR2019,Wheat,18/10/2019,Robin,4.7




raw.inputs


,input_id,farm_id,application_date,input_type,quantity,unit,cost_kes
0,I00001,F0001,2020-03-23,manure,300.0,Kg,1247
1,I00002,F0001,2024-04-18,DAP 18-46-0,30.0,kgs,4263
2,I00003,F0001,31/05/2024,CAN,24.0,kg,None
3,I00004,F0002,2019-03-14,DAP 18-46-0,410.0,kg,None
4,I00005,F0002,02/05/2019,Pesticide,7.5,litres,8521




raw.harvests


,harvest_id,farm_id,season,crop,harvest_date,yield_qty,yield_unit,notes
0,H00001,F0001,2023 LR,beans,24/07/2023,0.5,bags,None
1,H00002,F0001,2024 Long Rains,Maize,22/08/2024,5.7,bags,None
2,H00003,F0002,2019 Long Rains,Maize,2019-08-11,13.15,tonnes,None
3,H00004,F0002,2019 Short Rains,Wheat,19/02/2020,8.91,t,None
4,H00005,F0002,2020 LR,wheat,08/07/2020,6.94,MT,None




raw.weather


,date,county,rainfall_mm,temp_max_c,temp_min_c
0,2019-01-01,Kisumu,0.0,28.2,16.4
1,2019-01-02,Kisumu,0.0,27.7,15.5
2,2019-01-03,Kisumu,0.0,29.4,15.5
3,2019-01-04,Kisumu,0.0,29.9,16.6
4,2019-01-05,Kisumu,0.0,28.4,15.9


### How messy is the data?

> Now that we have loaded the data successfully into the db, let's preview the messiness of the data.



In [ ]:
# 1. Same crop name, different spelling formats.
conn.sql(
    """
      SELECT crop,
          COUNT(*) as count
      FROM raw.harvests
      GROUP BY 1
      ORDER BY 2 DESC
    """
).df()

,crop,count
0,Maize,1836
1,Beans,1028
2,Wheat,408
3,Sorghum,390
4,maize,294
5,MAIZE,161
6,beans,154
7,Corn,126
8,Maize,114
9,wheat,73


In [ ]:
# 2. Harvest yeilds in different formats
conn.sql(
    """
      SELECT yield_unit,
        COUNT(*) AS count
      FROM raw.harvests
      GROUP BY 1
      ORDER BY 2 DESC
    """
).df()

,yield_unit,count
0,bags,1642
1,kg,1039
2,Bags,568
3,bag,385
4,Kg,330
5,tonnes,282
6,90kg bag,212
7,KG,190
8,kgs,141
9,t,84


In [ ]:
# 3. Explore messiness in weather table
conn.sql(
    """
      SELECT rainfall_mm,
        COUNT(*) AS count
      FROM raw.weather
      WHERE TRY_CAST(rainfall_mm AS FLOAT) IS NULL OR TRY_CAST(rainfall_mm AS FLOAT) < 0
      GROUP BY 1
      ORDER BY 2 DESC
    """
).df()

,rainfall_mm,count
0,None,420
1,trace,135
2,-999,75


In [ ]:
conn.close()

This stage outputs an `agric_elt.duckdb` with fuve tables in the `raw` schema identical to the source CSV files. The `.duckdb` file is git-ignored since re-running the notebook rebuilds the file.

## Staging: Data Transformations

> The `raw` tables are too messy to be joined and analyzed reliably. This stage turns each `raw.*` table into a clean, analysis-ready `staging.*` table using SQL.

> This stage represents the T in 'ELT' and the tranformations happens inside the database without modifing the `raw` schema.





In [ ]:
conn.sql(
    """
      SELECT table_name
      FROM information_schema.tables
      WHERE table_schema == 'staging'
      ORDER BY 1
    """
).df()

,table_name


### Reusable Cleaning Functions

> There are fields that appear in all five tables eg fields IDs and dates; we will define SQL functions to reuse across all tables.



In [26]:
# Farms IDS: Standardise spelling formats e.g f0002, F-0002 and "F0002 " >> become F0002
conn.sql(
    """
      CREATE OR REPLACE MACRO clean_farm_id(x) AS
        upper(regexp_replace(trim(x), '[^A-Za-z0-9]', '', 'g'))
    """
)

# Correct misspelt crop names; aliases such as corn should be tranformed to maize
conn.sql(
    """
      CREATE OR REPLACE MACRO clean_crop(x) AS
        CASE lower(trim(x))
          WHEN 'corn' THEN 'maize'
          WHEN 'bean' THEN 'beans'
          WHEN 'sorgum' THEN 'sorghum'
          WHEN 'wheat grain' THEN 'wheat'
        END
    """
)

# Varying season labels:
# '2023 Long Rains', 'LR2023', '2023 LR' e.t.c >> become 2023-LR
conn.sql(
    """
      CREATE OR REPLACE MACRO clean_season(x) AS
        regexp_extract(x, '[0-9{4}]', 0) || '-' ||
        CASE WHEN lower(x) LIKE '%short%' OR upper(x) LIKE '%SR%' THEN 'SR'
             WHEN lower(x) LIKE '%long%' OR upper(x) LIKE '%LR%' THEN 'LR'
        END
    """
)

# Clean and standardize date formats
conn.sql(
    """
      CREATE OR REPLACE MACRO parse_date(x) AS
        CAST(COALESCE(
            TRY_STRPTIME(x, '%Y-%m-%d'),
            TRY_STRPTIME(x, '%d/%m/%Y'),
            TRY_STRPTIME(x, '%d-%b-%y'),
            TRY_STRPTIME(x, '%b %d, %Y')
        ) AS Date)
    """
)

# Quick test
conn.sql(
    """
      SELECT clean_farm_id('f-0005 ') AS farm_id,
        clean_crop('corn') AS crop,
        clean_season('2023 Long Rains') AS season,
        parse_date('14-Sep-26') AS date_1,
        parse_date('2021-03-21') AS date_2
    """
).df()

,farm_id,crop,season,date_1,date_2
0,F0005,maize,2-LR,2026-09-14,2021-03-21


### Transform `farms` table.

In [28]:
for name in tables:
  if name == 'farms':
    print(f'__raw.{name}__')
    display(conn.sql(f'SELECT * EXCLUDE (source_file, loaded_at) FROM raw.{name} LIMIT 5').df())

__raw.farms__


,farm_id,county,size_acres,soil_type,irrigated,registration_date
0,F-0001,Kakamega,0.6 acres,clay loam,no,29/03/2020
1,F0002,Uasin Gishu,11.1,loam,N,2012-05-06
2,F0003,Meru,1.0,clay,N,2021-04-14
3,F0004,Meru,1.3,Sandy,N,13/11/2008
4,F0005,Narok,6.2,clay loam,No,2021-03-29
